In [1]:
!pip install -q -U google-genai gspread google-auth google-auth-oauthlib google-api-python-client

In [2]:
from google.colab import auth
from google.colab import userdata
from googleapiclient.discovery import build
from google import genai
from google.genai import types
import gspread
import google.auth
import re
import smtplib
from email.mime.text import MIMEText
from datetime import datetime

auth.authenticate_user()

creds, project = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/documents",
    "https://www.googleapis.com/auth/gmail.send",
])

gc = gspread.authorize(creds)
drive_service = build("drive", "v3", credentials=creds)
docs_service  = build("docs",  "v1", credentials=creds)
gemini_client = genai.Client(api_key="Enter Your API Key")

SHEET_URL = "https://docs.google.com/spreadsheets/d/1DypIrUHP1uTp3-r1uEqq3u05yT7zBeMQkYqCV6cwt4o/edit"
sheet = gc.open_by_url(SHEET_URL).sheet1

print("Connected! All services are ready.")

Connected! All services are ready.


In [3]:
def generate_pantry_plan(timeline, budget, restrictions, gender, weight, age, height, county, email):

    system_instruction = """
You are a 'Biometric Financial Engineer' serving university students in Maryland.
You calculate survival grocery plans based on biometric caloric needs and strict financial amortization.

CRITICAL RULES:
1. ZERO FLUFF — Output ONLY the required Markdown structure below. No preamble, no sign-off.
2. GROUNDING — Use the google_search tool to find REAL grocery prices at stores in or near the
   user's Maryland county (Aldi, Walmart, Giant, Lidl, etc.).
3. LINK QUOTA — Provide EXACTLY 7 direct URLs to staple grocery items. No more, no less.
4. MATH VALIDATION — Calculate Daily Burn Rate (Total Budget / Days).
   The sum of all daily amortized food costs MUST be less than or equal to Daily Burn Rate. Show your math.
5. BIOMETRICS — Calculate TDEE using Mifflin-St Jeor. Make sure the plan meets minimum caloric needs.
6. DIETARY COMPLIANCE — All 7 items must respect the user's stated restrictions/allergies.

OUTPUT FORMAT — use EXACTLY these headers, no variations:

# MEAL PLAN REPORT
**Prepared For:** [email]
**Report Date:** [today's date]
**Plan Horizon:** [timeline]

---

# SECTION I — BIOMETRIC & FINANCIAL BASELINE
| Metric | Value |
|--------|-------|
| Estimated TDEE | [X] kcal/day |
| Total Budget | $[X] |
| Plan Duration | [X] days |
| Daily Burn Rate | $[X]/day |
| Dietary Restrictions | [restrictions or "None"] |

---

# SECTION II — AMORTIZED SHOPPING LIST
(Exactly 7 items. One per row.)

| # | Item | Store / Link | Unit Price | Units Needed | Total Cost | Daily Amortized Cost |
|---|------|-------------|-----------|-------------|-----------|---------------------|
| 1 | [name] | [store]([url]) | $X | X | $X | $X/day |
| | **TOTAL** | | | | **$X** | **$X/day** |

**Budget Compliance Check:** $[total daily cost] <= $[Daily Burn Rate]

---

# SECTION III — DAILY PACING STRATEGY
(Exactly what to eat each day to stretch all 7 staples across the full timeline.
Break it into morning / midday / evening meals. Keep it precise and actionable.)

---

# SECTION IV — RISK & VARIANCE NOTES
(Flag any nutritional gaps, substitution options if an item is out of stock,
and one cost-reduction tip.)
    """

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.1,
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )

    user_prompt = f"""
Calculate the amortization plan for the following student:

- Email: {email}
- Report Date: {datetime.today().strftime('%B %d, %Y')}
- Timeline: {timeline}
- Total Budget: {budget}
- Dietary Restrictions/Allergies: {restrictions if restrictions else 'None'}
- Gender: {gender}
- Weight: {weight} lbs
- Age: {age} years
- Height: {height}
- Location: {county} County, Maryland

Remember: EXACTLY 7 verified grocery links. Strict Markdown headers only. Verify all math.
    """

    print(f"  Gemini is searching live grocery prices in {county} County...")

    try:
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_prompt,
            config=config,
        )
        return response.text
    except Exception as e:
        return f"Error generating plan: {str(e)}"

In [9]:
COLOR_CHARCOAL = {"red": 0.114, "green": 0.122, "blue": 0.129}
COLOR_GOLD     = {"red": 0.831, "green": 0.686, "blue": 0.216}
COLOR_OFFWHITE = {"red": 0.922, "green": 0.914, "blue": 0.890}
COLOR_LTGRAY   = {"red": 0.600, "green": 0.600, "blue": 0.600}

def _range(start, end):
    return {"startIndex": start, "endIndex": end}

def build_investment_doc(markdown_text, email, doc_id):
    sections = {}
    current_section = None
    buffer = []

    # Parse the Markdown into sections
    for line in markdown_text.split("\n"):
        stripped = line.strip()
        if stripped.startswith("# "):
            if current_section and buffer:
                sections[current_section] = "\n".join(buffer).strip()
            current_section = stripped[2:].strip()
            buffer = []
        elif stripped == "---":
            continue
        else:
            buffer.append(line)

    if current_section and buffer:
        sections[current_section] = "\n".join(buffer).strip()

    today_str = datetime.today().strftime("%B %d, %Y")
    lines = []

    # Build Header
    lines.append("MEALPREP AI")
    lines.append("Nutritional Amortization Report")
    lines.append(f"Prepared for: {email}")
    lines.append(f"Report Date: {today_str}")
    lines.append("")

    section_order = [
        "SECTION I — BIOMETRIC & FINANCIAL BASELINE",
        "SECTION II — AMORTIZED SHOPPING LIST",
        "SECTION III — DAILY PACING STRATEGY",
        "SECTION IV — RISK & VARIANCE NOTES",
    ]

    section_labels = {
        "SECTION I — BIOMETRIC & FINANCIAL BASELINE":  "I.   Biometric & Financial Baseline",
        "SECTION II — AMORTIZED SHOPPING LIST":         "II.  Amortized Shopping List",
        "SECTION III — DAILY PACING STRATEGY":          "III. Daily Pacing Strategy",
        "SECTION IV — RISK & VARIANCE NOTES":           "IV.  Risk & Variance Notes",
    }

    # Populate Sections
    for key in section_order:
        label = section_labels.get(key, key)
        lines.append(label)
        lines.append("")
        body = sections.get(key, "[No data returned for this section.]")
        clean = []
        for ln in body.split("\n"):
            # Strip markdown bolding for the Doc
            ln = re.sub(r"\*\*(.+?)\*\*", r"\1", ln)
            clean.append(ln)
        lines.append("\n".join(clean))
        lines.append("")

    lines.append("-" * 60)
    lines.append("Auto-generated by MealPrep AI. Prices verified at time of generation.")

    full_text = "\n".join(lines)

    # Insert text into Doc
    docs_service.documents().batchUpdate(
        documentId=doc_id,
        body={"requests": [{"insertText": {"location": {"index": 1}, "text": full_text}}]},
    ).execute()

    # Retrieve inserted content to find formatting ranges
    doc_body = docs_service.documents().get(documentId=doc_id).execute()
    content  = doc_body.get("body", {}).get("content", [])

    para_ranges = []
    for element in content:
        para = element.get("paragraph")
        if not para:
            continue
        start = element.get("startIndex", 0)
        end   = element.get("endIndex", 0)
        text  = "".join(
            seg.get("textRun", {}).get("content", "")
            for seg in para.get("elements", [])
        ).rstrip("\n")
        para_ranges.append((start, end, text))

    def find_para(search_text):
        for s, e, t in para_ranges:
            if search_text in t:
                return s, e
        return None, None

    fmt = []

    # 1. Page Background & Style
    fmt.append({
        "updateDocumentStyle": {
            "documentStyle": {
                "pageSize": {
                    "width":  {"magnitude": 612, "unit": "PT"},
                    "height": {"magnitude": 792, "unit": "PT"},
                },
                "marginTop":    {"magnitude": 72, "unit": "PT"},
                "marginBottom": {"magnitude": 72, "unit": "PT"},
                "marginLeft":   {"magnitude": 72, "unit": "PT"},
                "marginRight":  {"magnitude": 72, "unit": "PT"},
                "background": {"color": {"color": {"rgbColor": COLOR_CHARCOAL}}},
            },
            "fields": "pageSize,marginTop,marginBottom,marginLeft,marginRight,background",
        }
    })

    # Helper Functions for Formatting
    def para_fmt(start, end, space_above=0, space_below=0, bg=None, align="START"):
        r = {
            "updateParagraphStyle": {
                "range": _range(start, end),
                "paragraphStyle": {
                    "spaceAbove": {"magnitude": space_above, "unit": "PT"},
                    "spaceBelow": {"magnitude": space_below, "unit": "PT"},
                    "alignment": align,
                },
                "fields": "spaceAbove,spaceBelow,alignment",
            }
        }
        if bg:
            r["updateParagraphStyle"]["paragraphStyle"]["shading"] = {
                "backgroundColor": {"color": {"rgbColor": bg}}
            }
            r["updateParagraphStyle"]["fields"] += ",shading"
        return r

    def text_fmt(start, end, size=11, font="Courier New", color=None, bold=False, italic=False):
        style = {
            "bold": bold, "italic": italic,
            "fontSize": {"magnitude": size, "unit": "PT"},
            "weightedFontFamily": {"fontFamily": font},
        }
        if color:
            style["foregroundColor"] = {"color": {"rgbColor": color}}
        return {
            "updateTextStyle": {
                "range": _range(start, end),
                "textStyle": style,
                "fields": "bold,italic,fontSize,weightedFontFamily" + (",foregroundColor" if color else ""),
            }
        }

    # Apply Header Styles
    s, e = find_para("MEALPREP AI")
    if s is not None and e > s:
        fmt.append(para_fmt(s, e, space_above=20, space_below=4, bg=COLOR_CHARCOAL, align="CENTER"))
        fmt.append(text_fmt(s, e - 1, size=26, font="Courier New", color=COLOR_GOLD, bold=True))

    s, e = find_para("Nutritional Amortization Report")
    if s is not None and e > s:
        fmt.append(para_fmt(s, e, space_above=0, space_below=6, bg=COLOR_CHARCOAL, align="CENTER"))
        fmt.append(text_fmt(s, e - 1, size=12, font="Courier New", color=COLOR_OFFWHITE, italic=True))

    for meta in [f"Prepared for: {email}", f"Report Date: {today_str}"]:
        s, e = find_para(meta)
        if s is not None and e > s:
            fmt.append(para_fmt(s, e, space_above=2, space_below=2, bg=COLOR_CHARCOAL, align="CENTER"))
            fmt.append(text_fmt(s, e - 1, size=9, font="Courier New", color=COLOR_LTGRAY))

    for key in section_labels.values():
        s, e = find_para(key)
        if s is not None and e > s:
            fmt.append(para_fmt(s, e, space_above=16, space_below=4, bg=COLOR_CHARCOAL))
            fmt.append(text_fmt(s, e - 1, size=10, font="Courier New", color=COLOR_GOLD, bold=True))

    # Apply Body Text Styles with safety checks for empty paragraphs
    headers_to_skip = list(section_labels.values()) + [
        "MEALPREP AI", "Nutritional Amortization Report",
        f"Prepared for: {email}", f"Report Date: {today_str}",
        "Auto-generated by MealPrep AI"
    ]

    for s, e, t in para_ranges:
        # SKIP if the paragraph is empty or just a newline to prevent 400 Errors
        if e <= s + 1:
            continue

        if not any(h in t for h in headers_to_skip):
            safe_end = max(s, e - 1)
            if safe_end > s:
                fmt.append(text_fmt(s, safe_end, size=9, font="Courier New", color=COLOR_OFFWHITE))

    # Execute Batch Updates in chunks
    chunk_size = 50
    for i in range(0, len(fmt), chunk_size):
        docs_service.documents().batchUpdate(
            documentId=doc_id,
            body={"requests": fmt[i: i + chunk_size]},
        ).execute()

    print("   Doc formatted successfully.")

In [11]:
def run_pipeline(row_data, row_num, col_map):

    email        = str(row_data.get("Email", "")).strip()
    timeline     = str(row_data.get("Timeline", "")).strip()
    budget       = str(row_data.get("Budget", "")).strip()
    restrictions = str(row_data.get("Restrictions", "")).strip()
    gender       = str(row_data.get("Gender", "")).strip()
    weight       = str(row_data.get("Weight", "")).strip()
    age          = str(row_data.get("Age", "")).strip()
    height       = str(row_data.get("Height", "")).strip()
    county       = str(row_data.get("County", "")).strip()

    markdown_text = generate_pantry_plan(
        timeline, budget, restrictions, gender,
        weight, age, height, county, email
    )

    if markdown_text.startswith("Error generating plan"):
        raise RuntimeError(markdown_text)

    today_str = datetime.today().strftime("%Y-%m-%d")
    doc = drive_service.files().create(
        body={
            "name": f"MealPrep Report — {email} — {today_str}",
            "mimeType": "application/vnd.google-apps.document",
        },
        fields="id,webViewLink"
    ).execute()
    doc_id   = doc["id"]
    doc_link = doc["webViewLink"]

    build_investment_doc(markdown_text, email, doc_id)

    drive_service.permissions().create(
        fileId=doc_id,
        body={"type": "anyone", "role": "reader"},
    ).execute()

    sender_email = "sample@gmail.com"  # change this to your Gmail
    app_password = userdata.get("GMAIL_APP_PASSWORD")

    email_body = f"""Hello,

Your MealPrep AI report is ready!

View it here: {doc_link}

What's inside:
- Your calorie needs (TDEE) based on your height, weight, age, and gender
- A 7-item shopping list with real prices from stores near you
- A daily eating schedule to stay on budget
- Notes on what to substitute if something is out of stock

-- MealPrep AI
    """

    msg = MIMEText(email_body)
    msg["To"]      = email
    msg["From"]    = sender_email
    msg["Subject"] = "Your MealPrep AI Report is Ready"

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(sender_email, app_password)
        smtp.sendmail(sender_email, email, msg.as_string())

    sheet.update_cell(row_num, col_map["Status"],  "Success")
    sheet.update_cell(row_num, col_map["DocLink"], doc_link)

    print(f"  Email sent to {email}")
    print(f"  Doc link: {doc_link}")
    return doc_link

In [12]:
def start_mealprep_ai():

    records = sheet.get_all_records()
    headers = sheet.row_values(1)
    col_map = {name: idx + 1 for idx, name in enumerate(headers)}

    required = ["Email", "Timeline", "Budget", "Restrictions",
                "Gender", "Weight", "Age", "Height", "County",
                "Status", "DocLink"]
    missing = [c for c in required if c not in col_map]
    if missing:
        print(f"Missing columns in sheet: {missing}")
        print(f"Columns we found: {headers}")
        return

    # find the first unprocessed row only — one at a time
    for i, row in enumerate(records):
        if str(row.get("Status", "")).strip():
            continue

        row_num = i + 2
        email   = str(row.get("Email", f"Row {row_num}"))
        print(f"New submission found: {email}")

        try:
            run_pipeline(row, row_num, col_map)
            print(f"Done. Plan sent to {email}")
        except Exception as exc:
            print(f"Error on row {row_num}: {exc}")
            sheet.update_cell(row_num, col_map["Status"], f"Error: {str(exc)[:80]}")

        return  # stop after one row — next check will pick up the next one

    print("No new submissions.")


start_mealprep_ai()

New submission found: sample@gmail.com
  Gemini is searching live grocery prices in Howard County...
   Doc formatted successfully.
  Email sent to sample@gmail.com
  Doc link: https://docs.google.com/document/d/1cILyupfoX7JJws7oEmHvDcMR-I8nZBDZVhppydiT1fI/edit?usp=drivesdk
Done. Plan sent to sample@gmail.com


In [ ]:
import time

print("Watching for new submissions... press stop to quit.\n")

while True:
    start_mealprep_ai()
    print("Sleeping 2 minutes...\n")
    time.sleep(120)

Watching for new submissions... press stop to quit.

New submission found: sample@gmail.com
  Gemini is searching live grocery prices in Howard county County...
Error on row 4: Error generating plan: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing.', 'status': 'RESOURCE_EXHAUSTED'}}
Sleeping 2 minutes...



KeyboardInterrupt: 